In [3]:
import numpy as np
import pandas as pd

# ==========================================
# ALL INPUTS — CHANGE ANYTHING HERE
# ==========================================
r = 0.0890          # Risk-free rate (8.90%)
T = 1.0             # 1-year contract
prod_cost = 386.08  # Production cost (LKR) — updated
SIMS = 10000
SEED = 42

# ==========================================
# BOOTSTRAP MONTE CARLO FUNCTION (PRICE ONLY)
# ==========================================
def bootstrap_mc_premium(S0, K, r, T, historical_log_returns, simulations=SIMS, seed=SEED):
    """
    Pure price-index insurance premium using Bootstrap Monte Carlo.
    No weather adjustment. Beta is fixed at 1.0.
    """
    months_to_simulate = int(round(12 * T))
    rng = np.random.default_rng(seed)
    draws = rng.choice(historical_log_returns,
                       size=(simulations, months_to_simulate),
                       replace=True)
    annual_log_return = draws.sum(axis=1)
    ST = S0 * np.exp(annual_log_return)
    payouts = np.maximum(K - ST, 0)
    premium = np.exp(-r * T) * payouts.mean()
    return premium

# ==========================================
# LOAD REAL DATA FROM CSV
# ==========================================
df = pd.read_csv("Data_Python.csv")
df = df.rename(columns={
    "YEAR": "Date",
    "Final_Price_RSS 1": "RSS1",
    "Final_Price_RSS 3": "RSS3"
})
df["RSS1"] = pd.to_numeric(df["RSS1"], errors="coerce")
df["RSS3"] = pd.to_numeric(df["RSS3"], errors="coerce")
df = df.dropna(subset=["RSS1", "RSS3"]).reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"].str.title(), format="%Y%b", errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
df = df[(df["RSS1"] > 0) & (df["RSS3"] > 0)].reset_index(drop=True)

df["rss1_log_return"] = np.log(df["RSS1"] / df["RSS1"].shift(1))
df["rss3_log_return"] = np.log(df["RSS3"] / df["RSS3"].shift(1))

rss1_historical_returns = df["rss1_log_return"].dropna().to_numpy()
rss3_historical_returns = df["rss3_log_return"].dropna().to_numpy()

# ==========================================
# GET LATEST PRICES AND HISTORICAL MEANS
# ==========================================
last_idx = df["Date"].idxmax()
S0_rss1 = df.loc[last_idx, "RSS1"]
S0_rss3 = df.loc[last_idx, "RSS3"]
hist_mean_rss1 = df["RSS1"].mean()
hist_mean_rss3 = df["RSS3"].mean()

print("=" * 75)
print(" TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO (PRICE ONLY)")
print("=" * 75)
print(f"\n RSS1 latest price (S0) : {S0_rss1:.2f} LKR")
print(f" RSS3 latest price (S0) : {S0_rss3:.2f} LKR")
print(f" RSS1 historical mean   : {hist_mean_rss1:.2f} LKR")
print(f" RSS3 historical mean   : {hist_mean_rss3:.2f} LKR")
print(f"\n Simulations : {SIMS:,}   |   Random seed : {SEED}")

# ==========================================
# RUN MCS FOR ALL SCENARIOS (PRICE ONLY)
# ==========================================
grades = ["RSS 1", "RSS 3"]
s0_map = {"RSS 1": S0_rss1, "RSS 3": S0_rss3}
hmean_map = {"RSS 1": hist_mean_rss1, "RSS 3": hist_mean_rss3}
returns_map = {"RSS 1": rss1_historical_returns, "RSS 3": rss3_historical_returns}

results = []
for grade in grades:
    S = s0_map[grade]
    h_mean = hmean_map[grade]
    scenarios = {
        '100% Market Price' : S,
        '95% Market Price'  : S * 0.95,
        'Historical Mean'   : h_mean,
        'Production Cost'   : prod_cost,
    }
    
    for label, K in scenarios.items():
        premium_base = bootstrap_mc_premium(
            S, K, r, T, returns_map[grade]
        )
        
        results.append({
            'Grade'                : grade,
            'Scenario'             : label,
            'Strike K (LKR)'       : round(K, 2),
            'MCS Base Premium (LKR)': round(premium_base, 4)
        })

# ==========================================
# PRINT RESULTS
# ==========================================
df_final = pd.DataFrame(results)
print("\n" + "=" * 75)
for grade in grades:
    print(f" ── {grade} ──")
    df_g = df_final[df_final['Grade'] == grade][
        ['Scenario', 'Strike K (LKR)', 'MCS Base Premium (LKR)']
    ]
    print(df_g.to_string(index=False))
    print()
print("=" * 75)
print(" NOTE: These are pure price-protection premiums (no weather adjustment)")
print("=" * 75)

 TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO (PRICE ONLY)

 RSS1 latest price (S0) : 700.00 LKR
 RSS3 latest price (S0) : 845.00 LKR
 RSS1 historical mean   : 425.27 LKR
 RSS3 historical mean   : 400.60 LKR

 Simulations : 10,000   |   Random seed : 42

 ── RSS 1 ──
         Scenario  Strike K (LKR)  MCS Base Premium (LKR)
100% Market Price          700.00                 43.4137
 95% Market Price          665.00                 31.8315
  Historical Mean          425.27                  1.0878
  Production Cost          386.08                  0.4454

 ── RSS 3 ──
         Scenario  Strike K (LKR)  MCS Base Premium (LKR)
100% Market Price          845.00                 35.2494
 95% Market Price          802.75                 23.7743
  Historical Mean          400.60                  0.0008
  Production Cost          386.08                  0.0000

 NOTE: These are pure price-protection premiums (no weather adjustment)


In [5]:
import numpy as np
import pandas as pd

# ==========================================
# ALL INPUTS — SAME AS BLACK-SCHOLES CODE
# ==========================================
r = 0.0890          # Risk-free rate (8.90%)
T = 1.0             # 1-year contract
prod_cost = 386.08  # Production cost (LKR)
SIMS = 10000
SEED = 42

# Fixed S0 and historical means (matching BS code)
S0_rss1 = 760.00        # Latest RSS1 price (LKR) — December 2024
S0_rss3 = 686.25        # Latest RSS3 price (LKR) — December 2024
hist_mean_rss1 = 320.25 # Historical mean RSS1 (LKR)
hist_mean_rss3 = 307.88 # Historical mean RSS3 (LKR)

# ==========================================
# BOOTSTRAP MONTE CARLO FUNCTION (PRICE ONLY)
# ==========================================
def bootstrap_mc_premium(S0, K, r, T, historical_log_returns, simulations=SIMS, seed=SEED):
    """
    Pure price-index insurance premium using Bootstrap Monte Carlo.
    No weather adjustment. Beta is fixed at 1.0.
    """
    months_to_simulate = int(round(12 * T))
    rng = np.random.default_rng(seed)
    draws = rng.choice(historical_log_returns,
                       size=(simulations, months_to_simulate),
                       replace=True)
    annual_log_return = draws.sum(axis=1)
    ST = S0 * np.exp(annual_log_return)
    payouts = np.maximum(K - ST, 0)
    premium = np.exp(-r * T) * payouts.mean()
    return premium

# ==========================================
# LOAD REAL DATA FROM CSV (FOR LOG RETURNS ONLY)
# ==========================================
df = pd.read_csv("Data_Python.csv")
df = df.rename(columns={
    "YEAR": "Date",
    "Final_Price_RSS 1": "RSS1",
    "Final_Price_RSS 3": "RSS3"
})
df["RSS1"] = pd.to_numeric(df["RSS1"], errors="coerce")
df["RSS3"] = pd.to_numeric(df["RSS3"], errors="coerce")
df = df.dropna(subset=["RSS1", "RSS3"]).reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"].str.title(), format="%Y%b", errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
df = df[(df["RSS1"] > 0) & (df["RSS3"] > 0)].reset_index(drop=True)

df["rss1_log_return"] = np.log(df["RSS1"] / df["RSS1"].shift(1))
df["rss3_log_return"] = np.log(df["RSS3"] / df["RSS3"].shift(1))

rss1_historical_returns = df["rss1_log_return"].dropna().to_numpy()
rss3_historical_returns = df["rss3_log_return"].dropna().to_numpy()

# ==========================================
# RUN MCS FOR ALL SCENARIOS (PRICE ONLY)
# ==========================================
grades = ["RSS 1", "RSS 3"]
s0_map = {"RSS 1": S0_rss1, "RSS 3": S0_rss3}
hmean_map = {"RSS 1": hist_mean_rss1, "RSS 3": hist_mean_rss3}
returns_map = {"RSS 1": rss1_historical_returns, "RSS 3": rss3_historical_returns}

results = []
for grade in grades:
    S = s0_map[grade]
    h_mean = hmean_map[grade]
    scenarios = {
        '100% Market Price' : S,
        '95% Market Price'  : S * 0.95,
        'Historical Mean'   : h_mean,
        'Production Cost'   : prod_cost,
    }

    for label, K in scenarios.items():
        premium_base = bootstrap_mc_premium(
            S, K, r, T, returns_map[grade]
        )

        results.append({
            'Grade'                 : grade,
            'Scenario'              : label,
            'Strike K (LKR)'        : round(K, 2),
            'MCS Base Premium (LKR)': round(premium_base, 4)
        })

# ==========================================
# PRINT RESULTS
# ==========================================
df_final = pd.DataFrame(results)
print("\n" + "=" * 75)
print(" TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO (PRICE ONLY)")
print("=" * 75)
print(f"\n RSS1 S0 : {S0_rss1:.2f} LKR  |  RSS3 S0 : {S0_rss3:.2f} LKR")
print(f" RSS1 Historical Mean : {hist_mean_rss1:.2f} LKR  |  RSS3 Historical Mean : {hist_mean_rss3:.2f} LKR")
print(f" Production Cost : {prod_cost:.2f} LKR")
print(f" Risk-free rate  : {r*100:.2f}%  |  T : {T} year")
print(f" Simulations : {SIMS:,}   |   Random seed : {SEED}")
print("\n" + "=" * 75)
for grade in grades:
    print(f" -- {grade} --")
    df_g = df_final[df_final['Grade'] == grade][
        ['Scenario', 'Strike K (LKR)', 'MCS Base Premium (LKR)']
    ]
    print(df_g.to_string(index=False))
    print()
print("=" * 75)
print(" NOTE: These are pure price-protection premiums (no weather adjustment)")
print("=" * 75)


 TIERED PRICE-INDEX INSURANCE — BOOTSTRAP MONTE CARLO (PRICE ONLY)

 RSS1 S0 : 760.00 LKR  |  RSS3 S0 : 686.25 LKR
 RSS1 Historical Mean : 320.25 LKR  |  RSS3 Historical Mean : 307.88 LKR
 Production Cost : 386.08 LKR
 Risk-free rate  : 8.90%  |  T : 1.0 year
 Simulations : 10,000   |   Random seed : 42

 -- RSS 1 --
         Scenario  Strike K (LKR)  MCS Base Premium (LKR)
100% Market Price          760.00                 47.1348
 95% Market Price          722.00                 34.5599
  Historical Mean          320.25                  0.0202
  Production Cost          386.08                  0.2061

 -- RSS 3 --
         Scenario  Strike K (LKR)  MCS Base Premium (LKR)
100% Market Price          686.25                 28.6271
 95% Market Price          651.94                 19.3078
  Historical Mean          307.88                  0.0000
  Production Cost          386.08                  0.0426

 NOTE: These are pure price-protection premiums (no weather adjustment)
